# TabPFN-3 thinking mode on your own dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Innixma/priorlabs_tabpfn_demo/blob/main/notebooks/02_bring_your_own_data.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-181717?logo=github)](https://github.com/Innixma/priorlabs_tabpfn_demo)

[Thinking mode](https://docs.priorlabs.ai/capabilities/thinking-mode) applies additional
inference-time computation on top of [TabPFN-3](https://priorlabs.ai/technical-reports/tabpfn-3)
to push prediction quality further, steered toward the metric you declare. It works for
both classification and regression, and it relies only on TabPFN -- no LLMs, no real
data, no internet search, and no other model involved.

This notebook is a template: pick one of three dataset cells (a demo classification
dataset, a demo regression dataset, or **your own CSV**), configure the thinking-mode
options, and compare TabPFN-3 with and without thinking on your data.

> **Runtime**: thinking-mode fits run on the TabPFN API (roughly five minutes at high
> effort), so no GPU is needed. Note the API's default quota of **20 thinking fits per
> month** -- requests beyond it return HTTP 429.

## Setup

The install cell below is collapsed -- expand it to see the details. It is skipped
outside Colab so it never overwrites a locally managed environment.

In [ ]:
# @title Install dependencies { display-mode: "form" }
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    !command -v uv >/dev/null || pip install -q uv
    !uv pip install -q --python {__import__('sys').executable} pandas scikit-learn tabpfn-client

    # The install may replace Colab's preinstalled numpy; the copy already loaded in this
    # kernel then no longer matches the files on disk and imports break. When that happens,
    # restart the runtime once (continue from the next cell after it reconnects).
    import importlib.metadata
    import numpy
    if importlib.metadata.version("numpy") != numpy.__version__:
        print("numpy changed -- restarting the Colab runtime; re-run FROM THE NEXT CELL when it reconnects.")
        import os
        os.kill(os.getpid(), 9)

## The dataset -- run exactly one of the next three cells

Each cell produces the same four variables (`X_train`, `X_test`, `y_train`, `y_test`)
plus `task_type`, so everything below works no matter which one you ran.

In [ ]:
# --- Option A: demo classification dataset (breast cancer, 569 rows, binary) ---
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
X_all, y_all = data.data, data.target
task_type = "classification"

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.33, random_state=0, stratify=y_all
)
print(f"{task_type}: train {X_train.shape}, test {X_test.shape}, classes {sorted(y_all.unique())}")

In [ ]:
# --- Option B: demo regression dataset (diabetes progression, 442 rows) ---
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

data = load_diabetes(as_frame=True)
X_all, y_all = data.data, data.target
task_type = "regression"

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.33, random_state=0)
print(f"{task_type}: train {X_train.shape}, test {X_test.shape}, target range [{y_all.min():.1f}, {y_all.max():.1f}]")

In [ ]:
# --- Option C: bring your own dataset ---
import pandas as pd
from sklearn.model_selection import train_test_split

CSV_PATH = "your_data.csv"       # path or URL to a CSV file
LABEL = "target"                 # name of the label column
TASK_TYPE = "classification"     # "classification" or "regression"
TEST_SIZE = 0.33                 # held-out fraction for evaluation

df = pd.read_csv(CSV_PATH)
X_all, y_all = df.drop(columns=[LABEL]), df[LABEL]
task_type = TASK_TYPE

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=0,
    stratify=y_all if task_type == "classification" else None,
)
print(f"{task_type}: train {X_train.shape}, test {X_test.shape}")
X_train.head(3)

## TabPFN access token

TabPFN-3 runs through the TabPFN API, unlocked by a free Prior Labs access token:

1. Sign up / log in at [ux.priorlabs.ai](https://ux.priorlabs.ai)
2. Accept the license at [ux.priorlabs.ai/account/licenses](https://ux.priorlabs.ai/account/licenses)
3. Copy your access token from [ux.priorlabs.ai/account](https://ux.priorlabs.ai/account)

Tip: save it as a Colab secret named `TABPFN_TOKEN` to skip the prompt next time.

In [ ]:
import getpass
import os

tabpfn_token = os.environ.get("TABPFN_TOKEN")
if not tabpfn_token:
    try:
        from google.colab import userdata
        tabpfn_token = userdata.get("TABPFN_TOKEN")
    except Exception:
        pass
while not tabpfn_token:
    tabpfn_token = getpass.getpass("Paste your TABPFN_TOKEN and press Enter: ").strip()
os.environ["TABPFN_TOKEN"] = tabpfn_token
print("token set")

## Configure thinking mode

The full option surface from the [docs](https://docs.priorlabs.ai/capabilities/thinking-mode):

| Option | Values | What it does |
|---|---|---|
| `thinking_mode` | `True` / `False` | Turns thinking on (setting `thinking_effort` also enables it) |
| `thinking_effort` | `"medium"` / `"high"` | How much inference-time compute to spend during fitting |
| `thinking_metric` | classification: `accuracy`, `log_loss`, `roc_auc` -- regression: `rmse`, `mae` | The metric the extra compute is steered toward |
| `thinking_timeout_s` | seconds | Optional wall-clock time budget |

Leave `THINKING_METRIC = None` to pick a sensible default for your task
(`roc_auc` for binary classification, `log_loss` for multiclass, `rmse` for regression) --
but if you know the metric your application is judged on, declare it.

In [ ]:
import tabpfn_client
from tabpfn_client import TabPFNClassifier, TabPFNRegressor

tabpfn_client.set_access_token(os.environ["TABPFN_TOKEN"])

THINKING_EFFORT = "high"     # "medium" or "high"
THINKING_METRIC = None       # see the table above; None picks a default for your task
THINKING_TIMEOUT_S = None    # optional wall-clock budget in seconds

if THINKING_METRIC is None:
    if task_type == "classification":
        THINKING_METRIC = "roc_auc" if y_train.nunique() == 2 else "log_loss"
    else:
        THINKING_METRIC = "rmse"

model_cls = TabPFNClassifier if task_type == "classification" else TabPFNRegressor
thinking_kwargs = dict(thinking_mode=True, thinking_effort=THINKING_EFFORT, thinking_metric=THINKING_METRIC)
if THINKING_TIMEOUT_S is not None:
    thinking_kwargs["thinking_timeout_s"] = THINKING_TIMEOUT_S
print(f"{model_cls.__name__}({', '.join(f'{k}={v!r}' for k, v in thinking_kwargs.items())})")

## Baseline -- TabPFN-3 out of the box

A plain fit first, so the thinking-mode gain on your data is visible. The helper reports
every relevant metric for the task type.

In [ ]:
import time

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)


def evaluate(model, X_test, y_test, task_type):
    if task_type == "classification":
        proba = model.predict_proba(X_test)
        pred = np.asarray(model.classes_)[np.argmax(proba, axis=1)]
        scores = {
            "accuracy": accuracy_score(y_test, pred),
            "log_loss": log_loss(y_test, proba, labels=model.classes_),
        }
        if len(model.classes_) == 2:
            y_bin = (np.asarray(y_test) == model.classes_[1]).astype(int)
            scores["roc_auc"] = roc_auc_score(y_bin, proba[:, 1])
        return scores
    pred = model.predict(X_test)
    return {
        "rmse": mean_squared_error(y_test, pred) ** 0.5,
        "mae": mean_absolute_error(y_test, pred),
        "r2": r2_score(y_test, pred),
    }


baseline = model_cls()
t0 = time.time()
baseline.fit(X_train, y_train)
baseline_scores = evaluate(baseline, X_test, y_test, task_type)
print(f"TabPFN-3 (fit {time.time() - t0:.0f}s): "
      + "   ".join(f"{k}={v:.4f}" for k, v in baseline_scores.items()))

## Thinking mode

Same model, same data, thinking on. This is the cell that spends a thinking fit from
your monthly quota.

In [ ]:
thinking = model_cls(**thinking_kwargs)
t0 = time.time()
thinking.fit(X_train, y_train)
thinking_scores = evaluate(thinking, X_test, y_test, task_type)
print(f"TabPFN-3 (thinking, fit {time.time() - t0:.0f}s): "
      + "   ".join(f"{k}={v:.4f}" for k, v in thinking_scores.items()))

print()
print(f"{'metric':<10} {'baseline':>10} {'thinking':>10}")
for k in baseline_scores:
    print(f"{k:<10} {baseline_scores[k]:>10.4f} {thinking_scores[k]:>10.4f}")

## Notes for practice

- Declare the metric your application is judged on via `thinking_metric` -- thinking mode
  steers its extra inference-time computation toward it.
- Thinking mode trades inference-time compute for prediction quality; reach for it when
  the last points of a metric are valuable. Reserve quota for high-ROI datasets (default
  is 20 thinking fits per month).
- Thinking mode composes with TabPFN-3's native text-feature support, so a single call
  can handle mixed numerical, categorical, and text columns under the same
  inference-time-compute regime -- your CSV's text columns can stay as they are.
- It achieves this relying only on TabPFN -- no LLMs, no real data, no internet search,
  and no other model involved.
- It runs through the TabPFN API (`tabpfn-client`), so it works from any laptop -- no GPU
  required.